# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional Lane Selection: Lane 2 — Refresh / Content Opportunity Scoring**

I have selected **Lane 2: Refresh / Content Opportunity Scoring**.

**Why this lane?**
Prioritizing content refreshes is a fundamental, high-impact business problem for FlyRank and its clients. In search engine optimization (SEO), content decays over time, causing search visibility and organic traffic to decline. Editors and SEO teams have constrained resources and cannot inspect or update every page. By building an opportunity scoring and ranking system, we can prioritize pages that have significant search volume but are experiencing traffic decay and have not been recently updated. In our starter dataset of 30,000 pages, more than half of the search visibility is associated with declining pages, representing a substantial risk of traffic loss. Targeting this lane over the next 7 weeks offers a direct opportunity to deliver high-leverage business value by driving traffic recovery.

In [1]:
selected_lane = 'Lane 2: Refresh / Content Opportunity Scoring'
print(f'Provisional Lane: {selected_lane}')

Provisional Lane: Lane 2: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

**The Research Question:**
*Which high-demand pages that are experiencing visibility decline should content editors prioritize for refresh/update to recover or protect search traffic?*

- **Decision to Improve:** "Which specific subset of pages in the client's content inventory should a content editor review and update first?"
- **Action Taken:** The content editor / SEO strategist will review the prioritized pages and take one of several actions: revise/expand the content (e.g., rewrite outdated sections, add fresh statistics, improve intent alignment), adjust metadata (titles/meta descriptions), or prune/redirect the page if it is no longer useful.
- **Cost of a Wrong Recommendation:**
  - *False Positive (recommending a page that doesn't need refresh or won't benefit)*: Wasted editor hours (editing a page that was fine or has low recovery potential).
  - *False Negative (missing a page that is declining with high demand)*: Continued traffic loss, leading to a permanent drop in search visibility, session engagement, and conversion revenue for the client.
- **Why Data / ML is Needed:**
  - *Scale*: Manual auditing of tens of thousands of pages (up to 500,000+ pages in the warehouse) is impossible for human teams.
  - *Complexity*: A simple heuristic rule (e.g., "refresh every page older than 180 days") is too crude. It ignores current search demand, the severity of the decline, CTR anomalies, and user engagement levels.
  - *Interacting Signals*: Machine learning can ingest and weigh multiple interacting signals (age, GSC impressions, CTR, average position, GA4 engagement rate, and AI referred sessions) to predict decline risk and rank candidates by potential opportunity.

In [2]:
decision = "Which specific subset of pages in the client's content inventory should a content editor review and update first?"
action = "Revising, expanding, or pruning content to recover or protect traffic"
cost_false_positive = "Wasted editor hours on pages that do not benefit from a refresh"
cost_false_negative = "Continued traffic and revenue decay due to unaddressed content decay"

print(f"Decision to improve: {decision}")
print(f"Action to take: {action}")
print(f"Cost of False Positive: {cost_false_positive}")
print(f"Cost of False Negative: {cost_false_negative}")

Decision to improve: Which specific subset of pages in the client's content inventory should a content editor review and update first?
Action to take: Revising, expanding, or pruning content to recover or protect traffic
Cost of False Positive: Wasted editor hours on pages that do not benefit from a refresh
Cost of False Negative: Continued traffic and revenue decay due to unaddressed content decay


## 3. Quick look at the data (2-3 real numbers)

**Quick look at the data (supporting numbers from the starter dataset):**
1. **54.21% of pages** (16,262 out of 30,000 pages) in the starter dataset are currently declining (`trend_direction == 'down'`).
2. **51.27% of total search visibility** (79,994,363 GSC impressions out of 156,010,989) is captured by these declining pages. This proves that search visibility decline is a major problem affecting more than half of the client's search exposure.
3. There are **3,340 pages** that are high-demand (>=1,000 GSC impressions), declining, and stale (have not been updated in >=90 days). This represents a large pool of high-impact opportunities that a prioritized refresh queue can target.

In [3]:
import pandas as pd
import numpy as np

# Load the starter dataset
# Note: The notebook is in work/notebooks, so data is relative to repo root: ../../data/raw/...
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Calculate real numbers
total_pages = len(df)
declining_pages = (df['trend_direction'] == 'down').sum()
pct_declining_pages = declining_pages / total_pages * 100

total_impressions = df['impressions_90d'].sum()
declining_impressions = df[df['trend_direction'] == 'down']['impressions_90d'].sum()
pct_declining_impressions = declining_impressions / total_impressions * 100

# High-demand, declining, and stale pages
high_demand_stale_declining = df[
    (df['trend_direction'] == 'down') &
    (df['impressions_90d'] >= 1000) &
    (df['days_since_last_update'] >= 90)
]
count_high_demand_stale_declining = len(high_demand_stale_declining)

print(f"1. Proportion of declining pages: {pct_declining_pages:.2f}% ({declining_pages:,} out of {total_pages:,} pages)")
print(f"2. Proportion of total search visibility (impressions) from declining pages: {pct_declining_impressions:.2f}% ({declining_impressions:,} out of {total_impressions:,} impressions)")
print(f"3. High-demand (>=1,000 GSC impressions), declining, stale (>=90 days since last update) pages: {count_high_demand_stale_declining:,}")

1. Proportion of declining pages: 54.21% (16,262 out of 30,000 pages)
2. Proportion of total search visibility (impressions) from declining pages: 51.27% (79,994,363 out of 156,010,989 impressions)
3. High-demand (>=1,000 GSC impressions), declining, stale (>=90 days since last update) pages: 3,340


## 4. Careful words: what I can and can't claim

**What I CAN claim:**
- **Observational Associations:** We can identify which features (e.g. content age, word count, GSC average position, GA4 engagement metrics) are historically correlated with traffic decline.
- **Decision-Support Prioritization:** We can build a ranking model that assists editors in prioritizing their workflow by sorting pages from highest to lowest opportunity.
- **Directional Trend Identification:** We can identify pages showing clear, sustained downward trends in search impressions and clicks over a defined window (e.g., last 30 days vs. previous 30 days).

**What I CANNOT claim:**
- **Causal Guarantee:** We cannot claim that refreshing a page is *guaranteed* to cause a recovery or stop the decline. To prove causality, we would need to run controlled A/B experiments or use causal inference frameworks.
- **Algorithm Reverse-Engineering:** We cannot claim to "predict Google's search algorithm" or reverse-engineer its internal ranking weights.
- **Semantic Optimization:** We cannot claim that we are optimizing the semantic meaning of the content, because our dataset does not contain raw text or keywords (only scrambled metadata hashes). We are optimizing purely based on structural and performance metrics.

In [4]:
claims_supported = [
    "Observational associations between content metrics and visibility decline",
    "Decision-support prioritization queue for editors",
    "Directional traffic trend identification (last 30d vs prev 30d)"
]
claims_unsupported = [
    "Causal proof that a content refresh causes traffic recovery",
    "Reverse engineering Google's search ranking algorithm",
    "Semantic or content meaning-based optimization (due to anonymized query hashes)"
]

print("CAN CLAIM:")
for claim in claims_supported:
    print(f"  - {claim}")
print("\nCANNOT CLAIM:")
for claim in claims_unsupported:
    print(f"  - {claim}")

CAN CLAIM:
  - Observational associations between content metrics and visibility decline
  - Decision-support prioritization queue for editors
  - Directional traffic trend identification (last 30d vs prev 30d)

CANNOT CLAIM:
  - Causal proof that a content refresh causes traffic recovery
  - Reverse engineering Google's search ranking algorithm
  - Semantic or content meaning-based optimization (due to anonymized query hashes)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.